In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from typing import Optional
from keras import Sequential
from keras.layers import Flatten, Dropout, Dense
import re
import string
import unicodedata
import joblib
from google.colab import files

nltk.download("wordnet")
nltk.download("stopwords")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
data = pd.read_csv("twitter_training.csv")

In [ ]:
data.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [ ]:
data.shape

(74681, 4)

# Doing basic data analysis

In [ ]:
for i in data.columns:
  print(i,":")
  print(data[i].nunique())

2401 :
12447
Borderlands :
32
Positive :
4
im getting on borderlands and i will murder you all , :
69490


In [ ]:
# dropping column
data.drop("2401", axis= 1, inplace= True)

In [ ]:
list(set(data["Borderlands"].values))

['johnson&johnson',
 'CS-GO',
 'CallOfDuty',
 'Borderlands',
 'HomeDepot',
 'TomClancysRainbowSix',
 'Dota2',
 'Hearthstone',
 'AssassinsCreed',
 'Verizon',
 'MaddenNFL',
 'Xbox(Xseries)',
 'NBA2K',
 'PlayStation5(PS5)',
 'Microsoft',
 'WorldOfCraft',
 'Cyberpunk2077',
 'LeagueOfLegends',
 'Fortnite',
 'RedDeadRedemption(RDR)',
 'PlayerUnknownsBattlegrounds(PUBG)',
 'Overwatch',
 'Battlefield',
 'Facebook',
 'TomClancysGhostRecon',
 'FIFA',
 'Nvidia',
 'Google',
 'Amazon',
 'GrandTheftAuto(GTA)',
 'ApexLegends',
 'CallOfDutyBlackopsColdWar']

In [ ]:
# changing column name
data["text"]= data["im getting on borderlands and i will murder you all ,"]
data.drop("im getting on borderlands and i will murder you all ,",axis=1 , inplace= True)

In [ ]:
data["Sentiment"]= data["Positive"]
data.drop(["Positive"], axis= 1,inplace = True)

In [ ]:
data.head()

,Borderlands,text,Sentiment
0,Borderlands,I am coming to the borders and I will kill you...,Positive
1,Borderlands,im getting on borderlands and i will kill you ...,Positive
2,Borderlands,im coming on borderlands and i will murder you...,Positive
3,Borderlands,im getting on borderlands 2 and i will murder ...,Positive
4,Borderlands,im getting into borderlands and i can murder y...,Positive


In [ ]:
import re
import unicodedata
from typing import Optional

def clean_text(
    text: str,
    remove_extra_spaces: bool = True,
    remove_special_chars: bool = True,
    remove_numbers: bool = False,
    remove_punctuation: bool = False,
    lowercase: bool = False,
    remove_html_tags: bool = True,
    remove_urls: bool = True,
    remove_emails: bool = False,
    normalize_unicode: bool = True,
    remove_accents: bool = False,
    remove_newlines: bool = True,
    min_word_length: int = 0,
    language: str = 'bengali'  # 'bengali', 'english', 'mixed'
) -> str:

    if not isinstance(text, str):
        text = str(text)

    if remove_html_tags:
        text = re.sub(r'<[^>]+>', '', text)

    html_entities = {
        '&amp;': '&',
        '&lt;': '<',
        '&gt;': '>',
        '&quot;': '"',
        '&apos;': "'",
        '&#39;': "'",
        '&nbsp;': ' ',
    }
    for entity, char in html_entities.items():
        text = text.replace(entity, char)

    if remove_urls:
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z]{2,}', '', text)

    if remove_emails:
        text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '', text)

    if normalize_unicode:
        text = unicodedata.normalize('NFKD', text)

    if remove_accents:
        text = ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )

    if remove_newlines:
        text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    if remove_numbers:
        text = re.sub(r'\d+', '', text)

    if remove_special_chars:
        if language == 'bengali':
            text = re.sub(r'[^।-ঃ\u0980-\u09FFa-zA-Z0-9\s]', '', text)
        elif language == 'english':
            text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        else:  # mixed
            text = re.sub(r'[^।-ঃ\u0980-\u09FFa-zA-Z0-9\s]', '', text)

    if remove_punctuation:
        punctuation = r'[।,.!?;:\'\"`]'
        text = re.sub(punctuation, '', text)

    if remove_extra_spaces:
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()

    if lowercase:
        text = text.lower()

    if min_word_length > 0:
        words = text.split()
        words = [word for word in words if len(word) >= min_word_length]
        text = ' '.join(words)
    text= text.replace("not good","bad")
    text= text.replace("not bad","a bit good")
    return text.lower()

In [ ]:
data["Clean_text"]=data["text"].apply(clean_text)

In [ ]:
data.head()

,Borderlands,text,Sentiment,Clean_text
0,Borderlands,I am coming to the borders and I will kill you...,Positive,i am coming to the borders and i will kill you...
1,Borderlands,im getting on borderlands and i will kill you ...,Positive,im getting on borderlands and i will kill you all
2,Borderlands,im coming on borderlands and i will murder you...,Positive,im coming on borderlands and i will murder you...
3,Borderlands,im getting on borderlands 2 and i will murder ...,Positive,im getting on borderlands 2 and i will murder ...
4,Borderlands,im getting into borderlands and i can murder y...,Positive,im getting into borderlands and i can murder y...


In [ ]:
data.drop("Borderlands", axis= 1, inplace= True)

In [ ]:
data.head()

,text,Sentiment,Clean_text
0,I am coming to the borders and I will kill you...,Positive,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you ...,Positive,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...,Positive,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...,Positive,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...,Positive,im getting into borderlands and i can murder y...


In [ ]:
oe= OrdinalEncoder()
data["Sentiment"]= oe.fit_transform(data[["Sentiment"]])

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.drop_duplicates(inplace= True)

In [ ]:
x = data["Clean_text"]
y = data[["Sentiment"]]

In [ ]:
tf_idf= TfidfVectorizer()
count_vec = CountVectorizer()

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size= .20, random_state= 43)

In [ ]:
xtrain_tf_idf= tf_idf.fit_transform(xtrain)
xtrain_count = count_vec.fit_transform(xtrain)

In [ ]:
knn= KNeighborsClassifier(n_neighbors=9)
lr = LogisticRegression()
rand= RandomForestClassifier(n_estimators=50)

In [ ]:
ytrain.value_counts()

In [ ]:
model_knn = knn.fit(xtrain_tf_idf,ytrain)
model_lr = lr.fit(xtrain_tf_idf,ytrain)

In [ ]:
model_knn_cnt = knn.fit(xtrain_count,ytrain)
model_lr_cnt = lr.fit(xtrain_count,ytrain)

In [ ]:
model_rand= rand.fit(xtrain_tf_idf,ytrain)
model_rand_cnt= rand.fit(xtrain_count,ytrain)

In [ ]:
xtrain_count

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 915978 stored elements and shape (55817, 36662)>

In [ ]:
model_ann= Sequential()
model_ann.add(Dense(input_dim=55817,units= 128*2,activation= "relu"))
model_ann.add(Dropout(.2))
model_ann.add(Dense(128,activation= "relu"))
model_ann.add(Dropout(.2))
model_ann.add(Dense(64,activation= "relu"))
model_ann.add(Dropout(.2))
model_ann.add(Dense(32,activation= "relu"))
model_ann.add(Dropout(.2))
model_ann.add(Dense(16,activation= "relu"))
model_ann.add(Dropout(.2))
model_ann.add(Flatten())
model_ann.add(Dense(4,activation= "relu"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model_ann.compile(optimizer="adam",loss="mse",metrics=["mae"])
model_ann.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_13 (Dense)                │ (None, 256)            │    14,289,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,333,236 (54.68 MB)

 Trainable params: 14,333,236 (54.68 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_ann.fit(xtrain_count,ytrain,validation_data=(xtest,ytest),epochs= 50,batch_size= 100)

Epoch 1/50


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense_13' is incompatible with the layer: expected axis -1 of input shape to have value 55817, but received input with shape (None, 36662)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 36662), dtype=int64)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
models= {
    "model_knn_count":model_knn_cnt,
    "model_knn":model_knn,
    "model_lr":model_lr,
    "model_lr_count":model_lr_cnt,
    "model_rand_for":model_rand,
    "model_rand_cnt":model_rand_cnt
}

In [ ]:
for name,model in models.items():
  pred= model.predict(tf_idf.transform(xtest))
  print(f"--------------------{name}-------------------")
  print(classification_report(ytest,pred))

# Deciding Logistic Regression (with TF IDF vectorizer) as our model

In [ ]:
model_lr.predict(tf_idf.transform([clean_text("I'm there")]))

In [ ]:
y.value_counts()

In [ ]:
%%writefile app.py

In [ ]:
!python app.py

In [5]:
data["Positive"].value_counts()

,count
Positive,
Negative,22542
Positive,20831
Neutral,18318
Irrelevant,12990


In [7]:
model= joblib.load("model_lr.joblib")

In [8]:
model.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [9]:
!pip install -q pyngrok

In [20]:
from pyngrok import ngrok

ngrok.set_auth_token("3GXCdNXqjtOdQXg0cd78hc0SBMT_2NTBTT7NDR6gqysmb6ak6")
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://promotion-vengeful-marauding.ngrok-free.dev" -> "http://localhost:8501"


In [21]:
!streamlit run app.py

2026-08-11 15:03:53.928 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.237.244.7:8501

2026-08-11 15:04:39.150 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-08-11 15:04:51.740 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...
  Stopping...


In [22]:
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>